# VN30 – 5-year price forecasting project (29 tickers available)

**Source files:** CafeF historical-price Excel files supplied by the user.  
**Raw period:** 2021-11-18 to 2026-11-17.  
**Target:** next trading session's unadjusted closing price (`Close`).  
**Development:** target dates before 2025-11-18. **Final holdout:** target dates from 2025-11-18 through 2026-11-17.

Important: the uploaded archive contains 29 distinct tickers, not 30. Do not describe this as a complete 30-ticker dataset until the missing ticker's full five-year file set is supplied.

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
ROOT=Path.cwd()
DATA=ROOT/'data/processed/market_data_5y_29_tickers.csv'
df=pd.read_csv(DATA,parse_dates=['Date'])
print(df.shape, df.Ticker.nunique(), df.Date.min(), df.Date.max())
df.head()

## 1. Data quality and coverage

In [ ]:
print('Duplicate ticker-date:',df.duplicated(['Ticker','Date']).sum())
print(df[['Open','High','Low','Close','Matched_Volume']].isna().sum())
display(df.groupby('Ticker').agg(rows=('Date','size'),start=('Date','min'),end=('Date','max')))

## 2. EDA example: FPT

In [ ]:
g=df[df.Ticker=='FPT'].sort_values('Date')
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(g.Date,g.Close); ax.set_title('FPT closing price'); ax.grid(); plt.show()
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(g.Date,g.Matched_Volume); ax.set_title('FPT matched volume'); plt.show()
fig,ax=plt.subplots(figsize=(8,4)); ax.hist(g.Close,bins=30); ax.set_title('Close distribution'); plt.show()
fig,ax=plt.subplots(figsize=(8,3)); ax.boxplot(g.Close,vert=False); ax.set_title('Close boxplot'); plt.show()
sns.heatmap(g[['Open','High','Low','Close','Matched_Volume']].corr(),annot=True,cmap='coolwarm',center=0); plt.show()

## 3. Feature engineering and chronological holdout

Features use information available at the close of day *t* to predict the next trading session's Close. `Target_Date` records the date of that next session. The final year is separated using the **target date**, preventing the boundary observation from leaking into the wrong split.

In [ ]:
def make_features(g):
    g=g.sort_values('Date').copy()
    g['Price_Range']=g.High-g.Low
    g['Daily_Return']=g.Close.pct_change()
    g['MA5']=g.Close.rolling(5,min_periods=5).mean()
    g['MA20']=g.Close.rolling(20,min_periods=20).mean()
    g['Lag_1']=g.Close.shift(1)
    g['Lag_2']=g.Close.shift(2)
    g['Target_Date']=g.Date.shift(-1)
    g['Target_Next_Close']=g.Close.shift(-1)
    return g
features=['Open','High','Low','Close','Matched_Volume','Price_Range','Daily_Return','MA5','MA20','Lag_1','Lag_2']
sup=pd.concat([make_features(g) for _,g in df.groupby('Ticker')],ignore_index=True)
sup=sup.replace([np.inf,-np.inf],np.nan).dropna(subset=features+['Target_Next_Close','Target_Date'])
cutoff=pd.Timestamp('2025-11-18')
development=sup[sup.Target_Date<cutoff]
final_test=sup[sup.Target_Date>=cutoff]
print('Development:',development.shape,'Final test:',final_test.shape)

## 4. Baseline Linear Regression

A chronological validation segment is carved from the last 20% of the four-year development period. After validation, the baseline is refit on all four development years and evaluated once on the untouched final year. StandardScaler is fitted only on the applicable training partition.

In [ ]:
results=[]; predictions=[]
for ticker,g in sup.groupby('Ticker'):
    g=g.sort_values('Target_Date')
    dev=g[g.Target_Date<cutoff]; test=g[g.Target_Date>=cutoff]
    val_start=dev.Target_Date.quantile(.8)
    tr=dev[dev.Target_Date<val_start]; val=dev[dev.Target_Date>=val_start]
    s=StandardScaler(); Xtr=s.fit_transform(tr[features]); Xv=s.transform(val[features])
    model=LinearRegression().fit(Xtr,tr.Target_Next_Close)
    val_pred=model.predict(Xv)
    s2=StandardScaler(); Xd=s2.fit_transform(dev[features]); Xt=s2.transform(test[features])
    model2=LinearRegression().fit(Xd,dev.Target_Next_Close)
    yhat=model2.predict(Xt)
    results.append({'Ticker':ticker,'Development_rows':len(dev),'Final_Test_rows':len(test),
       'MAE':mean_absolute_error(test.Target_Next_Close,yhat),
       'MSE':mean_squared_error(test.Target_Next_Close,yhat),
       'RMSE':mean_squared_error(test.Target_Next_Close,yhat)**.5,
       'R2':r2_score(test.Target_Next_Close,yhat)})
    q=test[['Date','Target_Date','Ticker','Target_Next_Close']].copy(); q['Predicted_Next_Close']=yhat; predictions.append(q)
metrics=pd.DataFrame(results); pred=pd.concat(predictions,ignore_index=True)
display(metrics)
display(metrics[['MAE','MSE','RMSE','R2']].mean())

## 5. Final test charts and export

In [ ]:
f=pred[pred.Ticker=='FPT']
plt.figure(figsize=(12,4)); plt.plot(f.Target_Date,f.Target_Next_Close,label='Actual'); plt.plot(f.Target_Date,f.Predicted_Next_Close,label='Predicted'); plt.legend(); plt.show()
OUT=ROOT/'reports'; OUT.mkdir(exist_ok=True)
metrics.to_csv(OUT/'metrics_reproduced.csv',index=False)
pred.to_csv(OUT/'predictions_reproduced.csv',index=False)
print('Saved outputs to reports/')

## 6. Interpretation and limitations

- Current archive contains 29 tickers. One additional ticker with a full five-year history is required before calling this a 30-ticker project.
- Two OHLC range anomalies were found and flagged in the quality report; original values were preserved pending source verification.
- `Close` (not Adjusted Close) is used as requested. Corporate actions may create discontinuities in raw close prices; consider a separate adjusted-price sensitivity analysis if the instructor permits.
- RMSE/MAE are in each stock's price units and are not directly comparable across tickers.
- This is a baseline model, not evidence of a profitable trading strategy. Compare with a naive persistence forecast and ensemble models in the next stage.